# AGENTS026 – Hours 5–6: Remediation Planning & Execution Agents

This notebook extends the RCA step into action. It creates:
- a `RemediationPlanningAgent` that converts an `RCAResult` plus policy rules into an ordered list of `Action` objects,
- an `ActionExecutorAgent` with mock infrastructure functions,
- and an orchestrator `handle_incident(incident_id)` that ties RCA, planning, and execution together.

The emphasis here is safe automation: some actions may be auto-executable, while others require approval based on policy.


## Section 1 – Install / Import Dependencies

We continue using PydanticAI with the local vLLM OpenAI-compatible endpoint.


In [1]:
%pip install -q pydantic-ai-slim openai pandas pyarrow

print('Installed / ensured pydantic-ai-slim, openai, pandas, pyarrow')


Note: you may need to restart the kernel to use updated packages.
Installed / ensured pydantic-ai-slim, openai, pandas, pyarrow


In [2]:
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any
import os
import json
import asyncio
import uuid

import pandas as pd
from pydantic import BaseModel, Field

from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

print('Imports OK')


Imports OK


## Section 2 – Load Prior Outputs

We load incident candidates and RCA results created in the earlier notebooks.


In [3]:
root = Path.cwd() / 'agents026'
data_dir = root / 'data'
incidents_dir = data_dir / 'incidents'
exec_dir = data_dir / 'execution'
exec_dir.mkdir(parents=True, exist_ok=True)

incident_candidates_path = incidents_dir / 'incident_candidates.json'
rca_results_path = incidents_dir / 'rca_results.json'

with open(incident_candidates_path, 'r', encoding='utf-8') as f:
    incident_candidates_raw = json.load(f)

with open(rca_results_path, 'r', encoding='utf-8') as f:
    rca_results_raw = json.load(f)

len(incident_candidates_raw), len(rca_results_raw)


(4, 2)

## Section 3 – Recreate Schemas

Define the data models used by planning and execution.


In [4]:
class IncidentCandidate(BaseModel):
    incident_id: str
    start_time: datetime
    end_time: datetime
    services: List[str]
    anomaly_type: str
    metric_summary: Dict[str, float] = Field(default_factory=dict)
    log_samples: List[str] = Field(default_factory=list)
    k8s_event_samples: List[str] = Field(default_factory=list)
    change_refs: List[str] = Field(default_factory=list)

class RCAResult(BaseModel):
    incident_id: str
    root_cause_hypothesis: str
    impacted_components: List[str]
    probable_trigger: Optional[str] = None
    evidence: List[str] = Field(default_factory=list)
    confidence: float = Field(..., ge=0.0, le=1.0)

class Action(BaseModel):
    action_id: str = Field(default_factory=lambda: f'act-{uuid.uuid4().hex[:8]}')
    action_type: str
    target: str
    description: str
    parameters: Dict[str, object] = Field(default_factory=dict)
    requires_approval: bool = True
    status: str = 'pending'
    expected_impact: Optional[str] = None
    rationale: Optional[str] = None

class ActionPlan(BaseModel):
    incident_id: str
    actions: List[Action]

class ExecutionResult(BaseModel):
    incident_id: str
    executed_actions: List[Action]
    skipped_actions: List[Action]
    notes: List[str] = Field(default_factory=list)

incident_candidates = [IncidentCandidate(**x) for x in incident_candidates_raw]
incident_map = {x.incident_id: x for x in incident_candidates}
rca_results = [RCAResult(**x) for x in rca_results_raw]
rca_map = {x.incident_id: x for x in rca_results}

list(rca_map.keys())[:5]


['inc-001', 'inc-002']

## Section 4 – Client Configuration for vLLM

Use the same local OpenAI-compatible vLLM endpoint as the previous notebook.


In [5]:
BASE_URL = os.environ.get('BASE_URL', 'http://localhost:8000/v1')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'abc-123')
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen3-30B-A3B')

provider = OpenAIProvider(base_url=BASE_URL, api_key=OPENAI_API_KEY)
chat_model = OpenAIChatModel(MODEL_NAME, provider=provider)

print({'BASE_URL': BASE_URL, 'MODEL_NAME': MODEL_NAME})


{'BASE_URL': 'http://localhost:8000/v1', 'MODEL_NAME': 'Qwen3-30B-A3B'}


## Section 5 – Policy Rules

These simple policy rules determine whether an action can be auto-executed or requires approval. You can tune them later for the demo.


In [6]:
POLICY_RULES = {
    'auto_approve_action_types': [
        'restart_service',
        'scale_service',
        'clear_cache',
    ],
    'approval_required_action_types': [
        'rollback_deploy',
        'rollback_config',
        'disable_feature_flag',
        'drain_node',
    ],
    'max_auto_scale_replicas': 2,
    'min_confidence_for_aggressive_action': 0.75,
}

POLICY_RULES


{'auto_approve_action_types': ['restart_service',
  'scale_service',
  'clear_cache'],
 'approval_required_action_types': ['rollback_deploy',
  'rollback_config',
  'disable_feature_flag',
  'drain_node'],
 'max_auto_scale_replicas': 2,
 'min_confidence_for_aggressive_action': 0.75}

## Section 6 – RemediationPlanningAgent

This agent converts an RCA result plus policy rules into an ordered `ActionPlan`.

We ask it to:
- keep actions operationally plausible,
- order them from safest to strongest,
- and mark approval-sensitive actions correctly.


In [19]:
class PlanningDeps(BaseModel):
    incident_map: Dict[str, IncidentCandidate]
    rca_map: Dict[str, RCAResult]
    policy_rules: Dict[str, object]

planning_deps = PlanningDeps(incident_map=incident_map, rca_map=rca_map, policy_rules=POLICY_RULES)

PLANNING_SYSTEM_PROMPT = '''
You are a remediation planning agent for production incidents.
Given an RCA result and policy rules, create a safe, ordered list of remediation actions.
Rules:
1. Start with the least risky action that can reduce blast radius or gather safety.
2. If rollback or config reversal is likely needed, mark it as requires_approval unless policy clearly allows automatic execution.
3. Use concrete action types such as restart_service, scale_service, rollback_deploy, rollback_config, clear_cache, disable_feature_flag.
4. Every action must include target, description, parameters, expected_impact, rationale, and requires_approval.
5. Return ActionPlan only.
'''

planning_agent = Agent(
    model=chat_model,
    deps_type=PlanningDeps,
    output_type=ActionPlan,
    system_prompt=PLANNING_SYSTEM_PROMPT,
)

@planning_agent.tool
def get_rca_result(ctx: RunContext[PlanningDeps], incident_id: str) -> Dict[str, object]:
    return ctx.deps.rca_map[incident_id].model_dump()

@planning_agent.tool
def get_policy_rules(ctx: RunContext[PlanningDeps]) -> Dict[str, object]:
    return ctx.deps.policy_rules

@planning_agent.tool
def get_incident_context(ctx: RunContext[PlanningDeps], incident_id: str) -> Dict[str, object]:
    return ctx.deps.incident_map[incident_id].model_dump()

print('RemediationPlanningAgent ready')


RemediationPlanningAgent ready


In [20]:
class RemediationPlanningAgentWrapper:
    def __init__(self, agent: Agent, deps: PlanningDeps):
        self.agent = agent
        self.deps = deps

    def build_prompt(self, incident_id: str) -> str:
        return (
            f'Build a remediation plan for incident {incident_id}. '
            f'Use RCA result, policy rules, and incident context. Return ordered ActionPlan.'
        )

    async def plan_async(self, incident_id: str) -> ActionPlan:
        result = await self.agent.run(self.build_prompt(incident_id), deps=self.deps)
        return result.output

    def plan(self, incident_id: str) -> ActionPlan:
        return asyncio.run(self.plan_async(incident_id))

RemediationPlanningAgent = RemediationPlanningAgentWrapper(planning_agent, planning_deps)


## Section 7 – Mock Infrastructure Functions

These functions simulate infra actions. In the final demo, they provide clear, auditable execution logs without touching real systems.


In [21]:
mock_execution_log = []

def mock_restart_service(service: str) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated restart of service {service}'}
    mock_execution_log.append({'action': 'restart_service', 'target': service, **result})
    return result

def mock_scale_service(service: str, replicas_delta: int = 1) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated scale of {service} by +{replicas_delta} replicas'}
    mock_execution_log.append({'action': 'scale_service', 'target': service, 'replicas_delta': replicas_delta, **result})
    return result

def mock_clear_cache(service: str) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated cache clear for {service}'}
    mock_execution_log.append({'action': 'clear_cache', 'target': service, **result})
    return result

def mock_rollback_deploy(service: str, version: Optional[str] = None) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated rollback for {service} to previous stable version {version or "auto"}'}
    mock_execution_log.append({'action': 'rollback_deploy', 'target': service, 'version': version, **result})
    return result

def mock_rollback_config(service: str) -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated config rollback for {service}'}
    mock_execution_log.append({'action': 'rollback_config', 'target': service, **result})
    return result

def mock_disable_feature_flag(service: str, flag_name: str = 'unknown_flag') -> Dict[str, object]:
    result = {'status': 'executed', 'message': f'Simulated disabling feature flag {flag_name} for {service}'}
    mock_execution_log.append({'action': 'disable_feature_flag', 'target': service, 'flag_name': flag_name, **result})
    return result


## Section 8 – ActionExecutorAgent

This agent decides whether actions are executable under policy and routes them to the appropriate mock infra functions.

For demo safety, the actual infrastructure mutation is mocked, but the control flow is realistic.


In [22]:
class ExecutorDeps(BaseModel):
    policy_rules: Dict[str, object]

executor_deps = ExecutorDeps(policy_rules=POLICY_RULES)

EXECUTION_SYSTEM_PROMPT = '''
You are an action execution agent.
Evaluate each proposed action against policy.
Only execute actions that are safe and do not require approval.
If an action requires approval, leave it pending/skipped and explain why.
You do not invent new actions; you evaluate and route the given actions.
'''

executor_agent = Agent(
    model=chat_model,
    deps_type=ExecutorDeps,
    output_type=ExecutionResult,
    system_prompt=EXECUTION_SYSTEM_PROMPT,
)

@executor_agent.tool
def get_execution_policy(ctx: RunContext[ExecutorDeps]) -> Dict[str, object]:
    return ctx.deps.policy_rules

print('ActionExecutorAgent base ready')


ActionExecutorAgent base ready


In [23]:
class ActionExecutorAgentWrapper:
    def __init__(self, agent: Agent, deps: ExecutorDeps):
        self.agent = agent
        self.deps = deps

    def execute_action(self, action: Action) -> Action:
        if action.requires_approval:
            action.status = 'skipped'
            return action

        if action.action_type == 'restart_service':
            result = mock_restart_service(action.target)
            action.status = result['status']
        elif action.action_type == 'scale_service':
            replicas_delta = int(action.parameters.get('replicas_delta', 1))
            result = mock_scale_service(action.target, replicas_delta=replicas_delta)
            action.status = result['status']
        elif action.action_type == 'clear_cache':
            result = mock_clear_cache(action.target)
            action.status = result['status']
        elif action.action_type == 'rollback_deploy':
            version = action.parameters.get('version')
            result = mock_rollback_deploy(action.target, version=version)
            action.status = result['status']
        elif action.action_type == 'rollback_config':
            result = mock_rollback_config(action.target)
            action.status = result['status']
        elif action.action_type == 'disable_feature_flag':
            flag_name = str(action.parameters.get('flag_name', 'unknown_flag'))
            result = mock_disable_feature_flag(action.target, flag_name=flag_name)
            action.status = result['status']
        else:
            action.status = 'skipped'
        return action

    async def evaluate_and_execute_async(self, incident_id: str, actions: List[Action]) -> ExecutionResult:
        executed_actions = []
        skipped_actions = []
        notes = []

        for action in actions:
            if action.requires_approval:
                action.status = 'skipped'
                skipped_actions.append(action)
                notes.append(f'{action.action_id}: approval required for {action.action_type}')
                continue
            updated = self.execute_action(action)
            if updated.status == 'executed':
                executed_actions.append(updated)
            else:
                skipped_actions.append(updated)
                notes.append(f'{updated.action_id}: action not executed ({updated.action_type})')

        return ExecutionResult(
            incident_id=incident_id,
            executed_actions=executed_actions,
            skipped_actions=skipped_actions,
            notes=notes,
        )

    def evaluate_and_execute(self, incident_id: str, actions: List[Action]) -> ExecutionResult:
        return asyncio.run(self.evaluate_and_execute_async(incident_id, actions))

ActionExecutorAgent = ActionExecutorAgentWrapper(executor_agent, executor_deps)
print('ActionExecutorAgent wrapper initialized')


ActionExecutorAgent wrapper initialized


## Section 9 – Orchestrator: handle_incident(incident_id)

This function is the first full end-to-end orchestration step in the workflow.

Flow:
1. Retrieve RCA result for the incident.
2. Ask `RemediationPlanningAgent` for an ordered action plan.
3. Pass the plan to `ActionExecutorAgent`.
4. Persist outputs for later reporting/demo.


In [31]:
async def handle_incident(incident_id: str) -> Dict[str, object]:
    if incident_id not in rca_map:
        raise ValueError(f'No RCA result found for incident_id={incident_id}')

    plan = await RemediationPlanningAgent.plan_async(incident_id)
    execution = await ActionExecutorAgent.evaluate_and_execute_async(incident_id, plan.actions)

    bundle = {
        'incident_id': incident_id,
        'rca_result': rca_map[incident_id].model_dump(),
        'action_plan': plan.model_dump(),
        'execution_result': execution.model_dump(),
    }

    out_path = exec_dir / f'{incident_id}_workflow.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(bundle, f, default=str, indent=2)

    return bundle


## Section 10 – Test Planning Agent

First verify the remediation plan alone on one sample incident.


In [25]:
sample_incident_ids = list(rca_map.keys())[:2]
sample_incident_ids


['inc-001', 'inc-002']

In [26]:
if sample_incident_ids:
    sample_plan = await RemediationPlanningAgent.plan_async(sample_incident_ids[0])
    print(sample_plan.model_dump_json(indent=2))
else:
    print('No RCA outputs available; run the previous RCA notebook first.')


{
  "incident_id": "inc-001",
  "actions": [
    {
      "action_id": "act-98e7f79d",
      "action_type": "disable_feature_flag",
      "target": "catalog-api",
      "description": "Disable the recently activated feature flag (v2.2.2) to mitigate potential inefficiencies or errors introduced by the change.",
      "parameters": {
        "feature_flag_version": "v2.2.2"
      },
      "requires_approval": true,
      "status": "pending",
      "expected_impact": "May reduce error rates if the feature flag was causing issues, with minimal impact on service availability.",
      "rationale": "The RCA suggests the feature flag activation around 12:13 may have contributed to the error rate increase. Disabling it could help isolate and mitigate the issue quickly.', "
    }
  ]
}


## Section 11 – Test End-to-End Orchestration

Now run `handle_incident(incident_id)` for one or two incidents and inspect both the action plan and the execution results.


In [32]:
workflow_outputs = []
for iid in sample_incident_ids[:2]:
    try:
        bundle = await handle_incident(iid)
        workflow_outputs.append(bundle)
        print('\n===== WORKFLOW OUTPUT FOR', iid, '=====')
        print(json.dumps(bundle, indent=2)[:6000])
    except Exception as e:
        print('Failed for', iid, ':', repr(e))



===== WORKFLOW OUTPUT FOR inc-001 =====
{
  "incident_id": "inc-001",
  "rca_result": {
    "incident_id": "inc-001",
    "root_cause_hypothesis": "The error rate increase in catalog-api is likely due to a recent deployment (v2.1.8) and feature flag activation (v2.2.2) around 12:13-12:16. These changes may have introduced inefficiencies or errors in request handling, leading to higher error rates.",
    "impacted_components": [
      "catalog-api"
    ],
    "probable_trigger": "deploy",
    "evidence": [
      "Similar incidents (inc-003, inc-002) involved deployment changes and feature flags.",
      "Recent changes show a deploy at 12:16 and feature flag at 12:13.",
      "Error rates spiked after these changes."
    ],
    "confidence": 0.95
  },
  "action_plan": {
    "incident_id": "inc-001",
    "actions": [
      {
        "action_id": "action-001",
        "action_type": "scale_service",
        "target": "catalog-api",
        "description": "Scale down the catalog-api servi

## Section 12 – Execution Log Inspection

Review the mock infrastructure execution log. This is useful in the demo to show auditability and policy-aware automation.


In [33]:
pd.DataFrame(mock_execution_log) if mock_execution_log else pd.DataFrame(columns=['action','target','status','message'])


,action,target,replicas_delta,status,message
0,scale_service,catalog-api,1.0,executed,Simulated scale of catalog-api by +1 replicas
1,clear_cache,catalog-api,NaN,executed,Simulated cache clear for catalog-api
2,restart_service,catalog-api,NaN,executed,Simulated restart of service catalog-api
3,scale_service,catalog-api,1.0,executed,Simulated scale of catalog-api by +1 replicas


## Section 13 – Persist Summary Artifacts

Save a compact summary of workflow outputs so the next notebook can build reporting or a demo narrative from them.


In [34]:
summary_rows = []
for bundle in workflow_outputs:
    exec_result = bundle['execution_result']
    summary_rows.append({
        'incident_id': bundle['incident_id'],
        'planned_actions': len(bundle['action_plan']['actions']),
        'executed_actions': len(exec_result['executed_actions']),
        'skipped_actions': len(exec_result['skipped_actions']),
        'root_cause_hypothesis': bundle['rca_result']['root_cause_hypothesis'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = exec_dir / 'workflow_summary.parquet'
summary_df.to_parquet(summary_path, index=False)

print('Saved summary to:', summary_path)
summary_df


Saved summary to: /workspace/agents026/data/execution/workflow_summary.parquet


,incident_id,planned_actions,executed_actions,skipped_actions,root_cause_hypothesis
0,inc-001,4,2,2,The error rate increase in catalog-api is like...
1,inc-002,4,2,2,The error rate increase in catalog-api is like...
